# 🧪 HyperFrames POC — Colab T4

> **Mục tiêu:** Verify `npx hyperframes render` chạy headless trên Colab
> **Nếu fail:** Đổi hướng sang Remotion hoặc Puppeteer manual
> **Nếu pass:** Triển khai VIBE Phase A

In [ ]:
print('Cell 1: Cài đặt môi trường...')
!apt-get update -qq && apt-get install -y ffmpeg 2>&1 | tail -2
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash - 2>&1 | tail -2
!apt-get install -y nodejs 2>&1 | tail -2
!node --version && npm --version
print('OK: Node.js + FFmpeg ready')

In [ ]:
print('Cell 2: Tạo project test tối thiểu...')
import os, json

PROJ = '/content/test_project'
os.makedirs(PROJ, exist_ok=True)

# Audio: 5 giây im lặng
!ffmpeg -y -f lavfi -i anullsrc=r=24000:cl=mono -t 5 {PROJ}/audio.mp3 2>&1 | tail -2
print('Audio: 5s silent @ 24kHz')

# Scene HTML: 1 scene don gian
s1 = '''<!doctype html><html lang="vi"><head><meta charset="UTF-8"/></head>
<body style="margin:0;padding:0;background:#080B14;width:1080px;height:1920px;display:flex;align-items:center;justify-content:center;font-family:Arial,sans-serif">
<div style="text-align:center;color:white">
<h1 style="font-size:80px;font-weight:800;background:linear-gradient(135deg,#6366f1,#06b6d4);-webkit-background-clip:text;-webkit-text-fill-color:transparent">HYPERFRAMES POC</h1>
<p style="font-size:40px;color:#94a3b8;margin-top:40px">Colab T4 GPU + Headless Chrome</p>
</div>
</body></html>'''
with open(f'{PROJ}/s1.html', 'w') as f: f.write(s1)

# index.html: root timeline
idx = '''<!doctype html><html lang="vi"><head><meta charset="UTF-8"/><meta name="viewport" content="width=1080,height=1920"/></head>
<body style="margin:0;padding:0;background:#080B14;overflow:hidden;width:1080px;height:1920px">
<div id="root" data-composition-id="test" data-width="1080" data-height="1920" data-start="0" data-duration="5">
<audio id="my-audio" src="audio.mp3" data-start="0" data-duration="5" data-track-index="0" data-volume="1"></audio>
<div id="s1" data-composition-src="./s1.html" data-start="0" data-duration="5" data-track-index="1"></div>
</div>
<script src="https://cdn.jsdelivr.net/npm/gsap@3.14.2/dist/gsap.min.js"></script>
<script>window.__timelines=window.__timelines||{};window.__timelines["test"]=gsap.timeline({paused:!0});</script>
</body></html>'''
with open(f'{PROJ}/index.html', 'w') as f: f.write(idx)

print('Project created: index.html + s1.html + audio.mp3 (5s)')

In [ ]:
print('Cell 3: RUN HYPERFRAMES RENDER...')
import subprocess, sys, time

start = time.time()
result = subprocess.run(
    ['npx', '--yes', 'hyperframes@0.6.40', 'render', '/content/test_project', '--output', '/content/test_project/output.mp4'],
    capture_output=True, text=True, timeout=300, cwd='/content/test_project'
)
elapsed = time.time() - start
print(f'Exit code: {result.returncode}')
print(f'Time: {elapsed:.1f}s')

if result.returncode == 0 and os.path.exists('/content/test_project/output.mp4'):
    size = os.path.getsize('/content/test_project/output.mp4')
    print(f'\n🎉 SUCCESS! output.mp4: {size/1024:.1f} KB')
    print('\nHyperFrames WORKS on Colab!')
else:
    print('\nSTDOUT (last 500 chars):')
    print(result.stdout[-500:] if result.stdout else '(empty)')
    print('\nSTDERR (last 500 chars):')
    print(result.stderr[-500:] if result.stderr else '(empty)')
    print('\n❌ FAILED')